# Exercise 2b: Feature engineering

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [12]:
X_train = pd.read_csv("ex2_train.csv")
y_train = pd.read_csv("ex2_class_train.csv")
X_test = pd.read_csv("ex2_test.csv")
y_test = pd.read_csv("ex2_class_test.csv")

In [13]:
# define a utility function to print out the prediction performance
def evaluate_result(y_test, y_pred, clf):
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print(f'Precision: {precision_score(y_test, y_pred):.4f}')
    print(f'Recall: {recall_score(y_test, y_pred):.4f}')
    print(f'F1-score: {f1_score(y_test, y_pred):.4f}')
    print(f'AUC-ROC: {roc_auc_score(y_test, clf.predict_proba(X_test_processed)[:, 1]):.4f}')

## Prototyping (without feature engineering)

In [14]:
def preprocess(data_in):
    data = data_in.drop(columns=['Name'])
    
    data = data.fillna({
        'Age': data['Age'].median(),
        'Embarked': data['Embarked'].mode(dropna=True).iloc[0],
        'Fare': data['Fare'].median()
    })

    # Convert categorical variables to dummy/indicator variables
    data = pd.get_dummies(data, columns=['Sex', 'Embarked'], drop_first=True)

    return data

In [15]:
X_train_processed = preprocess(X_train)
X_test_processed = preprocess(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_processed, y_train.values.ravel())
y_pred = clf.predict(X_test_processed)

print('Random Forest Model without Feature Engineering')
evaluate_result(y_test, y_pred, clf)
# Save baseline metrics before clf and X_test_processed are replaced.
baseline_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'AUC-ROC': roc_auc_score(y_test, clf.predict_proba(X_test_processed)[:, 1])
}

Random Forest Model without Feature Engineering
Accuracy: 0.8101
Precision: 0.7778
Recall: 0.7568
F1-score: 0.7671
AUC-ROC: 0.8736


## Feature engineering

The classification using simple preprocessed data gives only mediocre performance.

**TODO: You should make use of the insights from your EDA (ex2a) to complete the following feature engineering function below.** Later the function will replace the simple preprocessing.

You will pass the exercise if your feature engineering can improve the performance (i.e., winning in three or more metrics).

In [19]:
# Selected using training-data cross-validation: age by title and class.
def extract_title(names):
    titles = names.str.extract(r',\s*([^.]*)\.', expand=False).str.strip()
    titles = titles.replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    return titles.where(titles.isin(['Mr', 'Miss', 'Mrs', 'Master']), 'Rare')


# Learn all imputation values from training data only.
age_median = X_train['Age'].median()
fare_median = X_train['Fare'].median()
embarked_mode = X_train['Embarked'].mode().iloc[0]
training_titles = extract_title(X_train['Name'])
age_by_title = X_train.groupby(training_titles)['Age'].median()
title_class = training_titles + '_' + X_train['Pclass'].astype(str)
age_groups = X_train.groupby(title_class)['Age'].agg(['median', 'count'])
# Use a title/class median only when at least five observed ages support it.
age_by_group = age_groups.loc[age_groups['count'] >= 5, 'median']


def feature_engineering(data_in):
    df = data_in.copy()
    df['Title'] = extract_title(df['Name'])

    # EDA: survival differed between singles, small and large families.
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['SmallFamily'] = df['FamilySize'].between(2, 4).astype(int)
    df['LargeFamily'] = (df['FamilySize'] >= 5).astype(int)

    # Imputation fallback: title/class median, title median, overall median.
    df['AgeMissing'] = df['Age'].isna().astype(int)
    keys = df['Title'] + '_' + df['Pclass'].astype(str)
    df['Age'] = df['Age'].fillna(keys.map(age_by_group))
    df['Age'] = df['Age'].fillna(df['Title'].map(age_by_title)).fillna(age_median)
    df['Fare'] = df['Fare'].fillna(fare_median)
    df['Embarked'] = df['Embarked'].fillna(embarked_mode)
    df['IsChild'] = (df['Age'] < 13).astype(int)

    # Fixed categories ensure matching columns for training and test data.
    categories = {
        'Sex': ['female', 'male', 'Unknown'],
        'Embarked': ['C', 'Q', 'S', 'Unknown'],
        'Title': ['Mr', 'Miss', 'Mrs', 'Master', 'Rare']
    }
    for column, levels in categories.items():
        fallback = 'Rare' if column == 'Title' else 'Unknown'
        values = df[column].where(df[column].isin(levels), fallback)
        df[column] = pd.Categorical(values, categories=levels)
    return pd.get_dummies(df.drop(columns=['Name']),
                          columns=list(categories), drop_first=True, dtype=int)

In [20]:
X_train_processed = feature_engineering(X_train)
X_test_processed = feature_engineering(X_test)

# Check the feature contract before training.
assert X_train_processed.columns.equals(X_test_processed.columns)
assert X_train_processed.index.equals(X_train.index)
assert X_test_processed.index.equals(X_test.index)
assert np.isfinite(X_train_processed.to_numpy(dtype=float)).all()
assert np.isfinite(X_test_processed.to_numpy(dtype=float)).all()

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_processed, y_train.values.ravel())
y_pred = clf.predict(X_test_processed)

print('Random Forest Model with Feature Engineering')
evaluate_result(y_test, y_pred, clf)

Random Forest Model with Feature Engineering
Accuracy: 0.8436
Precision: 0.8108
Recall: 0.8108
F1-score: 0.8108
AUC-ROC: 0.9066


In [21]:
# Compare full-precision scores; a tie does not count as an improvement.
engineered_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'AUC-ROC': roc_auc_score(y_test, clf.predict_proba(X_test_processed)[:, 1])
}
comparison = pd.DataFrame({
    'Baseline': baseline_metrics,
    'Feature engineering': engineered_metrics
})
comparison['Difference'] = comparison['Feature engineering'] - comparison['Baseline']
comparison['Improved'] = comparison['Difference'] > 0
print(comparison.to_string(float_format=lambda value: f'{value:.4f}'))
wins = int(comparison['Improved'].sum())
print(f'Improved metrics: {wins}/5')
print('Requirement met' if wins >= 3 else 'Requirement not met')

           Baseline  Feature engineering  Difference  Improved
Accuracy     0.8101               0.8436      0.0335      True
Precision    0.7778               0.8108      0.0330      True
Recall       0.7568               0.8108      0.0541      True
F1           0.7671               0.8108      0.0437      True
AUC-ROC      0.8736               0.9066      0.0330      True
Improved metrics: 5/5
Requirement met
